# CIC-IDS2017 — Preprocessing Strategy

## Objective

Define a reproducible preprocessing strategy for CIC-IDS2017 based on the findings from the preceding exploratory analyses.

This notebook does not perform final ML preprocessing or generate the final modelling dataset. Instead, it documents which data-quality issues require remediation, which features should be removed or reviewed, and which preprocessing strategies should be evaluated during model development.

### Inputs

- Dataset overview analysis
- Feature and data-quality analysis
- Class-distribution analysis
- Feature-distribution analysis

### Outputs

- Preprocessing decisions
- Features recommended for removal
- Features requiring further review
- Candidate modelling features
- Target-label mapping
- Preprocessing summary

## 1. Load Dataset

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/raw/cicids2017")
RESULTS_DIR = Path("../results/cicids2017/05_preprocessing")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"CSV files found: {len(csv_files)}")

for file in csv_files:
    print(file.name)

CSV files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [4]:
dataframes = []

for file in csv_files:
    df = pd.read_csv(file, low_memory=False)
    dataframes.append(df)

data = pd.concat(
    dataframes,
    ignore_index=True
)

print(f"Dataset shape: {data.shape}")

Dataset shape: (2830743, 79)


## 2. Normalize Column Names

In [5]:
data.columns = data.columns.str.strip()

print(data.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count

In [6]:
LABEL_COL = "Label"

assert LABEL_COL in data.columns, (
    f"Target column not found. Available columns: {data.columns.tolist()}"
)

print(f"Target column: {LABEL_COL}")

Target column: Label


In [7]:
print(f"Dataset shape: {data.shape}")
print(f"Columns: {len(data.columns)}")

Dataset shape: (2830743, 79)
Columns: 79


In [8]:
LABEL_COL = "Label"

assert LABEL_COL in data.columns

print(f"Target column: {LABEL_COL}")

Target column: Label


## 3. Establish Feature Groups

In [32]:
feature_columns = [
    col for col in data.columns
    if col != LABEL_COL
]

numeric_features = data[feature_columns].select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = data[feature_columns].select_dtypes(
    exclude=np.number
).columns.tolist()

print(f"Total features: {len(feature_columns)}")
print(f"Numerical features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Total features: 80
Numerical features: 78
Categorical features: 2


In [33]:
print("Target:", LABEL_COL)
print("Dataset shape:", data.shape)

Target: Label
Dataset shape: (2830743, 81)


## 4. Constant Features

In [10]:
constant_features = [
    col
    for col in feature_columns
    if data[col].nunique(dropna=False) <= 1
]

print(f"Constant features: {len(constant_features)}")
print(constant_features)

Constant features: 8
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [11]:
dropped_features = pd.DataFrame({
    "feature": constant_features,
    "reason": "Constant feature",
    "action": "DROP"
})

dropped_features

,feature,reason,action
0,Bwd PSH Flags,Constant feature,DROP
1,Bwd URG Flags,Constant feature,DROP
2,Fwd Avg Bytes/Bulk,Constant feature,DROP
3,Fwd Avg Packets/Bulk,Constant feature,DROP
4,Fwd Avg Bulk Rate,Constant feature,DROP
5,Bwd Avg Bytes/Bulk,Constant feature,DROP
6,Bwd Avg Packets/Bulk,Constant feature,DROP
7,Bwd Avg Bulk Rate,Constant feature,DROP


## 5. Duplicate Columns

In [12]:
duplicate_columns = []

for i, col1 in enumerate(feature_columns):
    for col2 in feature_columns[i + 1:]:
        if data[col1].equals(data[col2]):
            duplicate_columns.append((col1, col2))

duplicate_columns

[('Total Fwd Packets', 'Subflow Fwd Packets'),
 ('Total Backward Packets', 'Subflow Bwd Packets'),
 ('Fwd PSH Flags', 'SYN Flag Count'),
 ('Bwd PSH Flags', 'Bwd URG Flags'),
 ('Bwd PSH Flags', 'Fwd Avg Bytes/Bulk'),
 ('Bwd PSH Flags', 'Fwd Avg Packets/Bulk'),
 ('Bwd PSH Flags', 'Fwd Avg Bulk Rate'),
 ('Bwd PSH Flags', 'Bwd Avg Bytes/Bulk'),
 ('Bwd PSH Flags', 'Bwd Avg Packets/Bulk'),
 ('Bwd PSH Flags', 'Bwd Avg Bulk Rate'),
 ('Fwd URG Flags', 'CWE Flag Count'),
 ('Bwd URG Flags', 'Fwd Avg Bytes/Bulk'),
 ('Bwd URG Flags', 'Fwd Avg Packets/Bulk'),
 ('Bwd URG Flags', 'Fwd Avg Bulk Rate'),
 ('Bwd URG Flags', 'Bwd Avg Bytes/Bulk'),
 ('Bwd URG Flags', 'Bwd Avg Packets/Bulk'),
 ('Bwd URG Flags', 'Bwd Avg Bulk Rate'),
 ('Fwd Header Length', 'Fwd Header Length.1'),
 ('Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk'),
 ('Fwd Avg Bytes/Bulk', 'Fwd Avg Bulk Rate'),
 ('Fwd Avg Bytes/Bulk', 'Bwd Avg Bytes/Bulk'),
 ('Fwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk'),
 ('Fwd Avg Bytes/Bulk', 'Bwd Avg Bulk Rate

In [13]:
duplicate_column_report = pd.DataFrame(
    duplicate_columns,
    columns=["feature", "duplicate_of"]
)

duplicate_column_report

,feature,duplicate_of
0,Total Fwd Packets,Subflow Fwd Packets
1,Total Backward Packets,Subflow Bwd Packets
2,Fwd PSH Flags,SYN Flag Count
3,Bwd PSH Flags,Bwd URG Flags
4,Bwd PSH Flags,Fwd Avg Bytes/Bulk
5,Bwd PSH Flags,Fwd Avg Packets/Bulk
6,Bwd PSH Flags,Fwd Avg Bulk Rate
7,Bwd PSH Flags,Bwd Avg Bytes/Bulk
8,Bwd PSH Flags,Bwd Avg Packets/Bulk
9,Bwd PSH Flags,Bwd Avg Bulk Rate


## 6. Infinite Values

In [14]:
infinite_summary = pd.DataFrame({
    "infinite_count": [
        np.isinf(data[col].to_numpy()).sum()
        for col in numeric_features
    ]
}, index=numeric_features)

infinite_features = (
    infinite_summary[
        infinite_summary["infinite_count"] > 0
    ]
    .index
    .tolist()
)

infinite_summary.loc[infinite_features]

,infinite_count
Flow Bytes/s,1509
Flow Packets/s,2867


### Decision

Infinite numerical values will not be treated as separate valid observations during ML preprocessing.

They will be converted to missing values during the preprocessing pipeline and subsequently handled according to the selected missing-value strategy.

No rows are removed at this stage.

## 7. Missing Values

In [15]:
missing_summary = pd.DataFrame({
    "missing_count": data.isna().sum(),
    "missing_percentage": (
        data.isna().sum() / len(data) * 100
    )
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
]

missing_summary

,missing_count,missing_percentage
Flow Bytes/s,1358,0.047973


### Decision

Missing values are extremely limited in CIC-IDS2017.

Rather than dropping affected rows, missing numerical values will be handled through an explicit imputation step in the ML preprocessing pipeline.

Infinite values will first be converted to missing values so that both conditions are handled consistently.

## 8. Susicious Numerical Features

In [16]:
non_negative_features = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
]

In [18]:
non_negative_features = [
    col for col in non_negative_features
    if col in data.columns
]

In [19]:
suspicious_features = []

for col in non_negative_features:
    negative_count = (
        data[col] < 0
    ).sum()

    if negative_count > 0:
        suspicious_features.append({
            "feature": col,
            "negative_count": negative_count,
            "negative_percentage": (
                negative_count / len(data) * 100
            ),
            "action": "REVIEW"
        })

suspicious_features = pd.DataFrame(
    suspicious_features
)

suspicious_features

,feature,negative_count,negative_percentage,action
0,Flow Duration,115,0.004063,REVIEW
1,Flow Bytes/s,85,0.003003,REVIEW
2,Flow Packets/s,115,0.004063,REVIEW


## 9. Correlation-Based Feature Review

In [20]:
correlation_pairs = pd.read_csv(
    "../results/cicids2017/02_feature_data_quality/high_correlation_pairs.csv"
)

correlation_pairs.head()

,feature_1,feature_2,correlation
0,Total Fwd Packets,Subflow Fwd Packets,1.0
1,Fwd PSH Flags,SYN Flag Count,1.0
2,Fwd Header Length,Fwd Header Length.1,1.0
3,Fwd URG Flags,CWE Flag Count,1.0
4,Bwd Packet Length Mean,Avg Bwd Segment Size,1.0


In [21]:
high_corr_review = correlation_pairs.copy()

high_corr_review["action"] = "REVIEW"

### Decision

Highly correlated features will not be automatically removed solely on the basis of correlation.

Feature removal will be evaluated during model-oriented feature selection, considering:

- model type
- interpretability
- redundancy
- computational cost
- validation performance

Perfectly duplicated columns may be removed earlier where their redundancy is unambiguous.

## 10. Duplicate Rows

### Decision

Exact duplicate records represent a substantial portion of CIC-IDS2017 and require explicit handling before model training.

Duplicate records will be removed from the modelling dataset during final preprocessing.

However, duplicate analysis will remain documented separately so that the original dataset characteristics are preserved.

The final train/test strategy will also be designed to minimize leakage caused by highly similar or duplicated observations crossing dataset partitions.

In [22]:
duplicate_rows = data.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")

Duplicate rows: 308,381


## 11. Target Preparation

In [23]:
label_counts = (
    data[LABEL_COL]
    .value_counts(dropna=False)
    .sort_values(ascending=False)
)

label_counts

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

In [24]:
data["target_multiclass"] = (
    data[LABEL_COL]
    .astype(str)
    .str.strip()
)

In [25]:
data["target_binary"] = np.where(
    data["target_multiclass"].str.upper() == "BENIGN",
    "BENIGN",
    "ATTACK"
)

In [26]:
print(data["target_multiclass"].nunique())
print(data["target_binary"].value_counts())

15
target_binary
BENIGN    2273097
ATTACK     557646
Name: count, dtype: int64


## 12. Target Mapping

In [27]:
classes = sorted(
    data["target_multiclass"].unique()
)

target_mapping = pd.DataFrame({
    "class_name": classes,
    "class_id": range(len(classes))
})

target_mapping

,class_name,class_id
0,BENIGN,0
1,Bot,1
2,DDoS,2
3,DoS GoldenEye,3
4,DoS Hulk,4
5,DoS Slowhttptest,5
6,DoS slowloris,6
7,FTP-Patator,7
8,Heartbleed,8
9,Infiltration,9


## 13. Class Imbalance Strategy

## Class Imbalance Strategy

CIC-IDS2017 exhibits substantial class imbalance, including extremely rare attack classes.

No oversampling, undersampling, or class removal is performed during this preprocessing-strategy stage.

Candidate approaches for model development include:

- Class-weighted learning
- Controlled undersampling of dominant classes
- Oversampling of minority classes
- Synthetic sampling where appropriate

The final strategy should be evaluated using validation results and should preserve the ability to assess performance on rare attack categories.

Accuracy alone will not be considered sufficient for evaluating an IDS model.

## 14. Scaling Strategy

## Scaling Strategy

The numerical feature space exhibits substantial differences in scale, skewness, and range.

Scaling will therefore be treated as model-dependent:

- Tree-based models generally do not require feature scaling.
- Distance-based and gradient-based models may require scaling.
- Highly skewed features may require transformation before scaling.
- Robust scaling may be evaluated for features containing substantial extreme-value behaviour.

No transformation is permanently applied at this stage.

## 15. Candidate Feature Set

In [28]:
drop_features = set(constant_features)

candidate_features = [
    col
    for col in feature_columns
    if col not in drop_features
]

candidate_features_df = pd.DataFrame({
    "feature": candidate_features,
    "status": "CANDIDATE"
})

candidate_features_df

,feature,status
0,Destination Port,CANDIDATE
1,Flow Duration,CANDIDATE
2,Total Fwd Packets,CANDIDATE
3,Total Backward Packets,CANDIDATE
4,Total Length of Fwd Packets,CANDIDATE
...,...,...
65,Active Min,CANDIDATE
66,Idle Mean,CANDIDATE
67,Idle Std,CANDIDATE
68,Idle Max,CANDIDATE


## 16. Preprocessing Decision Table

In [29]:
preprocessing_decisions = pd.DataFrame([
    {
        "issue": "Column whitespace",
        "decision": "Normalize",
        "stage": "Initial preprocessing"
    },
    {
        "issue": "Infinite values",
        "decision": "Convert to NaN",
        "stage": "Initial preprocessing"
    },
    {
        "issue": "Missing values",
        "decision": "Impute",
        "stage": "ML pipeline"
    },
    {
        "issue": "Constant features",
        "decision": "Remove",
        "stage": "Feature preprocessing"
    },
    {
        "issue": "Duplicate columns",
        "decision": "Remove where identical",
        "stage": "Feature preprocessing"
    },
    {
        "issue": "Highly correlated features",
        "decision": "Review",
        "stage": "Feature selection"
    },
    {
        "issue": "Duplicate rows",
        "decision": "Investigate before final removal",
        "stage": "Dataset preparation"
    },
    {
        "issue": "Suspicious negative values",
        "decision": "Review",
        "stage": "Data validation"
    },
    {
        "issue": "Class imbalance",
        "decision": "Evaluate mitigation strategies",
        "stage": "Model training"
    },
    {
        "issue": "Feature scaling",
        "decision": "Model-dependent",
        "stage": "ML pipeline"
    }
])

preprocessing_decisions

,issue,decision,stage
0,Column whitespace,Normalize,Initial preprocessing
1,Infinite values,Convert to NaN,Initial preprocessing
2,Missing values,Impute,ML pipeline
3,Constant features,Remove,Feature preprocessing
4,Duplicate columns,Remove where identical,Feature preprocessing
5,Highly correlated features,Review,Feature selection
6,Duplicate rows,Investigate before final removal,Dataset preparation
7,Suspicious negative values,Review,Data validation
8,Class imbalance,Evaluate mitigation strategies,Model training
9,Feature scaling,Model-dependent,ML pipeline


## 17. Summary

In [34]:
preprocessing_summary = pd.DataFrame({
    "metric": [
        "Original rows",
        "Original columns",
        "Numerical features",
        "Categorical features",
        "Constant features",
        "Features with infinite values",
        "Duplicate rows",
        "Number of target classes",
    ],
    "value": [
        len(data),
        len(data.columns),
        len(numeric_features),
        len(categorical_features),
        len(constant_features),
        len(infinite_features),
        duplicate_rows,
        data["target_multiclass"].nunique()
    ]
})

preprocessing_summary

,metric,value
0,Original rows,2830743
1,Original columns,81
2,Numerical features,78
3,Categorical features,2
4,Constant features,8
5,Features with infinite values,2
6,Duplicate rows,308381
7,Number of target classes,15


## 18. Export

In [35]:
preprocessing_decisions.to_csv(
    RESULTS_DIR / "preprocessing_decisions.csv",
    index=False
)

dropped_features.to_csv(
    RESULTS_DIR / "dropped_features.csv",
    index=False
)

candidate_features_df.to_csv(
    RESULTS_DIR / "candidate_features.csv",
    index=False
)

suspicious_features.to_csv(
    RESULTS_DIR / "suspicious_features.csv",
    index=False
)

target_mapping.to_csv(
    RESULTS_DIR / "target_mapping.csv",
    index=False
)

preprocessing_summary.to_csv(
    RESULTS_DIR / "preprocessing_summary.csv",
    index=False
)

print("All preprocessing results exported successfully.")

All preprocessing results exported successfully.


## Conclusion

The preprocessing analysis establishes a clear strategy for preparing CIC-IDS2017 for subsequent machine-learning evaluation. The dataset requires explicit handling of infinite and missing values, removal of constant features, and consideration of substantial duplicate records and feature redundancy.

The analysis also confirms that class imbalance must be addressed during model development, particularly for the rare attack categories identified in the class-distribution analysis. However, no classes are removed or artificially balanced at this stage in order to preserve the original characteristics of the dataset.

Overall, CIC-IDS2017 remains a viable candidate for further ML evaluation, provided that the identified data-quality, redundancy, imbalance, and potential data-leakage issues are handled through a reproducible preprocessing and evaluation pipeline.

The resulting preprocessing decisions and candidate feature sets provide the basis for the next stage: **evaluating the practical ML suitability of CIC-IDS2017**.